# Projet · Réseaux de neurones : notebook modèle

**À rendre pour le 3 décembre 2026 · en binôme · Promeo FA27 INFO**

Ce notebook est un **squelette guidé**. Vous le copiez (`Fichier > Enregistrer une copie dans Drive`), vous suivez les étapes dans l'ordre et vous remplacez les `...` et les **✏️ À COMPLÉTER**. À la fin, il doit **s'exécuter de bout en bout** (`Exécution > Tout exécuter`) sans erreur : c'est un livrable.

### Les trois livrables
1. **Ce notebook Colab**, qui s'exécute de bout en bout.
2. **Un README d'une page** : le jeu de données, la question posée, le résultat principal, ce que vous avez appris.
3. **5 minutes de présentation** au début du jour 2 (3 décembre).

### Ce que j'évalue (barème indicatif sur 20)
| Critère | Points |
|---|---|
| Données et préparation : nettoyage, séparation entraînement / test | 3 |
| Un réseau et un entraînement qui fonctionnent | 4 |
| Au moins trois variantes comparées | 4 |
| Évaluation : accuracy, matrice de confusion, precision, recall, F1 | 4 |
| Analyse des erreurs et conclusions | 3 |
| Présentation orale et réponse individuelle | 2 |

*Ce projet sert d'entraînement pour l'évaluation du jour 5.*

### Plan conseillé (6 semaines)
| Semaine | À faire |
|---|---|
| 1 | Choisir le jeu de données, l'explorer (étapes 0 à 2) |
| 2 | Préparer les données (étape 3) |
| 3 | Un premier réseau qui fonctionne, la base de comparaison (étape 4) |
| 4 | Les trois variantes (étape 5) |
| 5 | Évaluation et analyse des erreurs (étapes 6 et 7) |
| 6 | README, répétition de la présentation, relecture |

---
## Étape 0 · Qui, quoi, pourquoi

**✏️ À COMPLÉTER** (deux phrases maximum par ligne).

In [ ]:
BINOME = "Prénom Nom · Prénom Nom"                      # ✏️
JEU_DE_DONNEES = "..."                                  # ✏️ nom et source (lien)
QUESTION = "..."                                        # ✏️ ex. « Peut-on reconnaître le type de vêtement à partir de l'image ? »
CLASSES = "..."                                         # ✏️ ce que le réseau doit prédire (et combien de classes)
print(BINOME, "·", JEU_DE_DONNEES)

### Choisir un jeu de données

| Option | Pour qui | Remarque |
|---|---|---|
| **Fashion-MNIST** (10 types de vêtements, images 28 × 28) | Démarrer sans préparation lourde | Même format que MNIST : prévoyez une analyse plus poussée |
| **Un jeu tabulaire** (UCI, Kaggle : fichier `.csv`) | Travailler la préparation | À vous de nettoyer, normaliser, choisir la cible |
| **CIFAR-10** (images couleur 32 × 32) | **Bonus** | Plus difficile : un réseau simple donnera une accuracy modeste |

**MNIST est exclu** (utilisé en TP). Sources : [archive.ics.uci.edu](https://archive.ics.uci.edu), [kaggle.com/datasets](https://www.kaggle.com/datasets). Pour un fichier `.csv`, importez-le dans Colab avec le dossier 📁 à gauche.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split

torch.manual_seed(0)
np.random.seed(0)
print("PyTorch", torch.__version__)

---
## Étape 1 · Charger les données

Gardez **une seule** des deux cellules (images **ou** tableau) et supprimez l'autre.

### Option A · Fashion-MNIST (images)

In [ ]:
from torchvision import datasets, transforms

tf = transforms.ToTensor()
train = datasets.FashionMNIST("data", train=True, download=True, transform=tf)
test = datasets.FashionMNIST("data", train=False, download=True, transform=tf)
NOMS_CLASSES = train.classes if hasattr(train, "classes") else [str(i) for i in range(10)]

# On réserve 10 000 images pour VALIDER les réglages ; le jeu de test ne sert qu'à la fin.
train, val = random_split(train, [50000, 10000], generator=torch.Generator().manual_seed(0))
train_loader = DataLoader(train, batch_size=64, shuffle=True)
val_loader = DataLoader(val, batch_size=1000)
test_loader = DataLoader(test, batch_size=1000)

X, y = next(iter(train_loader))
N_ENTREES, N_CLASSES = int(np.prod(X.shape[1:])), len(NOMS_CLASSES)
print(X.shape, "· entrées :", N_ENTREES, "· classes :", NOMS_CLASSES)

### Option B · Un jeu tabulaire (fichier `.csv`)

**✏️ À COMPLÉTER** : le nom du fichier, la colonne à prédire (la cible) et les colonnes inutiles.

In [ ]:
FICHIER = "mon_fichier.csv"          # ✏️
CIBLE = "nom_de_la_colonne_cible"    # ✏️
A_SUPPRIMER = []                     # ✏️ colonnes inutiles (identifiants, noms…)

df = pd.read_csv(FICHIER).drop(columns=A_SUPPRIMER)
print(df.shape)
df.head()

**Préparation d'un tableau, étape par étape** (à adapter) :
1. valeurs manquantes : `df.isna().sum()`, puis supprimer ou remplacer ;
2. colonnes texte : `pd.get_dummies(...)` pour les transformer en 0/1 ;
3. cible : la transformer en entiers `0, 1, 2…` (`pd.factorize`) ;
4. **séparer avant de normaliser** : on calcule moyenne et écart-type sur l'entraînement **seulement**, sinon des informations du test fuient dans l'entraînement ;
5. normaliser : (valeur − moyenne) / écart-type.

In [ ]:
from sklearn.model_selection import train_test_split

df = df.dropna()                                          # ✏️ adaptez le traitement des manquants
y_codes, NOMS_CLASSES = pd.factorize(df[CIBLE])
X_df = pd.get_dummies(df.drop(columns=[CIBLE])).astype("float32")

# 60 % entraînement · 20 % validation · 20 % test
X_tr, X_rest, y_tr, y_rest = train_test_split(X_df.values, y_codes, test_size=0.4, random_state=0, stratify=y_codes)
X_va, X_te, y_va, y_te = train_test_split(X_rest, y_rest, test_size=0.5, random_state=0, stratify=y_rest)

moyenne, ecart = X_tr.mean(axis=0), X_tr.std(axis=0) + 1e-8   # calculés sur l'ENTRAÎNEMENT
def vers_loader(X_, y_, shuffle):
    ds = TensorDataset(torch.tensor((X_ - moyenne) / ecart, dtype=torch.float32), torch.tensor(y_, dtype=torch.long))
    return DataLoader(ds, batch_size=64 if shuffle else 1000, shuffle=shuffle)

train_loader, val_loader, test_loader = vers_loader(X_tr, y_tr, True), vers_loader(X_va, y_va, False), vers_loader(X_te, y_te, False)
N_ENTREES, N_CLASSES = X_tr.shape[1], len(NOMS_CLASSES)
print("entrées :", N_ENTREES, "· classes :", list(NOMS_CLASSES))

---
## Étape 2 · Explorer : comprendre avant de modéliser

**✏️ À COMPLÉTER** : répondez par écrit dans la cellule texte qui suit.

À regarder : combien d'exemples par classe ? Les classes sont-elles équilibrées ? Y a-t-il des valeurs aberrantes ? À quoi ressemblent quelques exemples ?

In [ ]:
# Nombre d'exemples par classe dans l'entraînement
etiquettes = torch.cat([y for _, y in train_loader]).numpy()
effectifs = pd.Series(etiquettes).value_counts().sort_index()
effectifs.index = [str(NOMS_CLASSES[i]) for i in effectifs.index]
effectifs.plot(kind="bar", title="Exemples par classe (entraînement)")
plt.show()
print(effectifs)

**✏️ Vos observations** (3 à 5 lignes) : classes équilibrées ? données bruitées ? difficulté attendue ?

> ...

---
## Étape 3 · Les outils d'entraînement

Les fonctions du TP, un peu généralisées. **Ne les modifiez pas** sans raison : elles servent pour toutes vos variantes.

In [ ]:
def construire(activation="relu", n_cachees=32, n_couches=2):
    """Un réseau plein : N_ENTREES -> n_couches couches cachées -> N_CLASSES. Pas d'activation en sortie."""
    act = {"sigmoid": nn.Sigmoid, "relu": nn.ReLU, "tanh": nn.Tanh}[activation]
    couches, entree = [nn.Flatten()], N_ENTREES
    for _ in range(n_couches):
        couches += [nn.Linear(entree, n_cachees), act()]
        entree = n_cachees
    couches.append(nn.Linear(entree, N_CLASSES))
    return nn.Sequential(*couches)

def calculer_accuracy(model, loader):
    model.eval()
    bonnes = 0
    with torch.no_grad():
        for X, y in loader:
            bonnes += (model(X).argmax(dim=1) == y).sum().item()
    model.train()
    return bonnes / len(loader.dataset)

def entrainer(model, lr=0.1, epoques=10, verbose=True):
    """Boucle du cours. Renvoie le coût d'entraînement et l'accuracy de VALIDATION par époque."""
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    cout, acc_val, debut = [], [], time.time()
    for e in range(epoques):
        total, n = 0.0, 0
        for X, y in train_loader:
            pred = model(X)
            loss = loss_fn(pred, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item() * len(y); n += len(y)
        cout.append(total / n)
        acc_val.append(calculer_accuracy(model, val_loader))
        if verbose:
            print(f"Époque {e + 1}/{epoques} · coût {cout[-1]:.4f} · accuracy validation {acc_val[-1]:.1%}")
    return {"cout": cout, "acc_val": acc_val, "duree": time.time() - debut}

resultats = []
def noter(label, r, model, **reglages):
    resultats.append({"variante": label, **reglages,
                      "accuracy validation": round(r["acc_val"][-1], 4),
                      "coût final": round(r["cout"][-1], 4),
                      "réglages": sum(p.numel() for p in model.parameters()),
                      "durée (s)": round(r["duree"])})
    return pd.DataFrame(resultats)

def courbes(runs, titre=""):
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6))
    for nom, r in runs.items():
        a1.plot(r["cout"], label=nom); a2.plot(r["acc_val"], label=nom)
    a1.set_title("Coût d'entraînement"); a2.set_title("Accuracy de validation")
    for a in (a1, a2): a.set_xlabel("époque"); a.legend()
    fig.suptitle(titre); plt.show()

---
## Étape 4 · Un premier réseau : la base de comparaison

Avant de varier, il faut un point de départ qui **fonctionne**.

**Vérifications** : (1) le coût baisse ; (2) l'accuracy dépasse le hasard (1 / nombre de classes) ; (3) pas d'erreur de forme.

> Piège classique : un coût qui ne baisse pas vient presque toujours d'un `lr` mal choisi, de données non normalisées, ou d'une activation placée après la dernière couche.

In [ ]:
torch.manual_seed(0)
base = construire(activation="relu", n_cachees=32, n_couches=2)
run_base = entrainer(base, lr=0.1, epoques=10)
noter("base", run_base, base, activation="relu", cachees=32, couches=2, lr=0.1)

**✏️ À COMPLÉTER** : que valent le hasard et votre base ? Est-ce suffisant pour continuer ?

> Hasard : ... · base : ... · décision : ...

---
## Étape 5 · Au moins trois variantes comparées

Modifiez **une seule chose à la fois** par rapport à la base. Les pistes du cours : l'**activation**, le **taux d'apprentissage**, la **taille** (neurones par couche, nombre de couches). Vous pouvez en ajouter (nombre d'époques, `batch_size`…).

Pour **chaque variante**, écrivez avant de lancer : *hypothèse → résultat → explication*.

| Variante | Hypothèse (avant) | Résultat (après) | Explication |
|---|---|---|---|
| 1 | ... | ... | ... |
| 2 | ... | ... | ... |
| 3 | ... | ... | ... |

In [ ]:
def variante(label, activation="relu", n_cachees=32, n_couches=2, lr=0.1, epoques=10):
    torch.manual_seed(0)
    m = construire(activation, n_cachees, n_couches)
    r = entrainer(m, lr=lr, epoques=epoques, verbose=False)
    noter(label, r, m, activation=activation, cachees=n_cachees, couches=n_couches, lr=lr)
    print(f"{label} : accuracy validation {r['acc_val'][-1]:.1%} · coût final {r['cout'][-1]:.3f}")
    return m, r

# ✏️ À COMPLÉTER : trois variantes (changez UNE chose par rapport à la base)
m1, r1 = variante("variante 1 : ...", activation="sigmoid")        # ✏️
m2, r2 = variante("variante 2 : ...", lr=0.5)                        # ✏️
m3, r3 = variante("variante 3 : ...", n_cachees=128)                 # ✏️

courbes({"base": run_base, "variante 1": r1, "variante 2": r2, "variante 3": r3}, "Comparaison des variantes")
pd.DataFrame(resultats)

**✏️ À COMPLÉTER** : laquelle choisissez-vous, et pourquoi ? Une accuracy plus haute suffit-elle (durée, nombre de réglages, stabilité) ?

> ...

---
## Étape 6 · Évaluation finale

On choisit **une** variante (la meilleure sur la **validation**) et on la mesure **une seule fois** sur le jeu de **test** : ces données n'ont servi à aucun réglage, c'est ce qui rend la mesure honnête.

In [ ]:
modele_final = m1        # ✏️ la variante retenue (base, m1, m2 ou m3)

from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

def predire(model, loader):
    model.eval()
    vrais, predits = [], []
    with torch.no_grad():
        for X, y in loader:
            vrais.append(y); predits.append(model(X).argmax(dim=1))
    model.train()
    return torch.cat(vrais).numpy(), torch.cat(predits).numpy()

y_true, y_pred = predire(modele_final, test_loader)
print("accuracy TEST :", round((y_true == y_pred).mean(), 4))
print(classification_report(y_true, y_pred, target_names=[str(c) for c in NOMS_CLASSES], digits=3))

fig, ax = plt.subplots(figsize=(8, 6.5))
ConfusionMatrixDisplay.from_predictions(y_true, y_pred, display_labels=[str(c) for c in NOMS_CLASSES], ax=ax, cmap="Blues", colorbar=False, xticks_rotation=45)
plt.title("Matrice de confusion (test)")
plt.tight_layout(); plt.show()

**✏️ À COMPLÉTER**

- Accuracy, precision, recall et F1 : lesquelles sont pertinentes pour votre question, et pourquoi ?
- Quelle classe est la mieux reconnue (recall) ? La moins bien ?
- Y a-t-il un déséquilibre qui rend l'accuracy trompeuse ?

> ...

---
## Étape 7 · Analyse des erreurs

On regarde **ce qui est faux** et on cherche une explication. C'est la partie qui montre que vous avez compris, pas seulement exécuté.

In [ ]:
# Les confusions les plus fréquentes (hors diagonale)
cm = confusion_matrix(y_true, y_pred)
np.fill_diagonal(cm, 0)
paires = [(cm[i, j], i, j) for i in range(len(cm)) for j in range(len(cm)) if cm[i, j] > 0]
for n, i, j in sorted(paires, reverse=True)[:5]:
    print(f"{n:5d} fois : vrai « {NOMS_CLASSES[i]} » prédit « {NOMS_CLASSES[j]} »")

**✏️ À COMPLÉTER** : pour les 2 ou 3 confusions principales, proposez une explication (ressemblance visuelle, classes proches, données insuffisantes, bruit) et **une piste d'amélioration** que vous testeriez avec plus de temps.

> ...

Pour un jeu d'**images**, affichez quelques erreurs (adaptez le code du TP : `images_test`, `erreurs = np.where(y_true != y_pred)[0]`, `plt.imshow`).

---
## Étape 8 · Conclusion

**✏️ À COMPLÉTER** (10 lignes maximum) : question posée, meilleur résultat obtenu, variantes testées et ce qu'elles ont montré, limites de votre travail, ce que vous feriez ensuite.

> ...

---
## ✅ Avant de rendre : la liste de contrôle

- [ ] Le notebook s'exécute **de bout en bout** après `Exécution > Redémarrer et tout exécuter`.
- [ ] Les étapes **0 à 8** sont complétées, **aucun `...` ne reste**.
- [ ] Au moins **3 variantes** comparées, avec hypothèse, résultat et explication.
- [ ] Le jeu de test n'a été utilisé **qu'une fois**, à l'étape 6.
- [ ] Accuracy, matrice de confusion, precision, recall et F1 sont présentées **et commentées**.
- [ ] Le README d'une page est rédigé (jeu de données, question, résultat, enseignements).
- [ ] Les deux membres du binôme savent **expliquer chaque cellule** : la réponse individuelle est notée.
- [ ] Le lien de partage Colab est en **« Tous les utilisateurs disposant du lien »** (lecture).